# olaf_cookbook — copy-paste examples for the `OLAF` facade

> **Attach a lakehouse (or bind one with `%%configure`) before running any cell** — every
> action below resolves against the attached lakehouse. Without one, the first call fails
> with `UnsupportedOperationException — No default context`. See
> [fabric-import.md](../docs/fabric-import.md) for both binding paths.

Worked, one-per-cell examples of every interactive `OLAF.<action>(...)` call on the
self-contained [`olaf`](olaf.ipynb) runtime. Each cell shows the
exact params the action consumes, a one-line note on what it does, and an **example output** (as a
trailing comment) so you can see the shape before running. **You don't need this file to run the
framework** — `olaf.ipynb` alone runs every mode; this cookbook is just runnable examples.

> ⚠️ **Live-Fabric demo — NOT run in CI.** These cells call the live DAR REST API and read real
> control tables, so they only execute inside a Fabric workspace with `olaf` imported
> and a lakehouse attached. The CI test + coverage harness targets `olaf.ipynb` only;
> this cookbook is never executed or measured. Values below (`priya@contoso.com`, `SalesReaders`,
> `2026-07-15`, …) are illustrative placeholders. **The first cell is a guard** — running this notebook whole (Run All / a pipeline / `notebook.run`) exits immediately without touching anything; run the cells you want **one at a time**.

Every `OLAF.*` method returns a **Spark DataFrame** for an easy `display(...)` — a query method
returns its result table; an ops method returns a compact DataFrame *view* of the outcome envelope
(the raw dict stays at `OLAF.last_result`).

## Contents

- **Load & configure** — `%run olaf` + `OLAF.configure(...)`
- **Deployment** — `explain` (dry preview) · `generate` · `validate` (zero-write dry-run) ·
  `plan` · `apply` (full-replace / incremental) · `rollback`
- **Audit** — `show` (table / role / member) · `trace` · `grants` · `provenance` · `timeline` · `report` ·
  compliance & live-DAR (`coverage` · `effective_access` · `who_can_access` · `drift`) ·
  config time-travel & lineage (`table_history` · `at` · `config_diff` · `value_history`)
- **Maintenance** — `setup` · `health` · `status` · `diagnose_member`


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  STOP — this is an EXAMPLES notebook, not one to run whole.
#  The cells below call LIVE operations (generate / apply / rollback / setup) against the
#  attached lakehouse. Running this notebook end-to-end — Run All, a Data Factory pipeline,
#  or notebookutils.notebook.run — would execute them for real. So this guard runs FIRST and
#  exits immediately on any whole-notebook run (the %run below never even loads).
#
#  To use the cookbook: run the cells you want ONE AT A TIME, deliberately (start at the
#  %run cell), or copy the ones you need into your own notebook.
# ─────────────────────────────────────────────────────────────────────────────
import notebookutils

notebookutils.notebook.exit(
    "olaf_cookbook is examples-only — run cells one at a time, not the whole notebook"
)

Load the runtime. `%run` is Fabric's import: it defines every function/class + the `OLAF`
facade in this notebook's namespace. The ▶️ Run cell is guarded, so `%run` dispatches
NOTHING — `OLAF` is ready, exactly like `from olaf import OLAF`. (The pipeline path never
`%run`s; it passes `mode`.)

> A `%run` cell must hold **nothing else** — no comments, no code. Fabric rejects the cell
> with `MagicUsageError: %run cannot run with other code or magic commands.` That is why this
> explanation is a markdown cell rather than a comment above the magic.

In [ ]:
%run olaf

In [ ]:
# Optional: set base params shared by all three OLAF namespaces, so you don't repeat them on
# every call. Anything the parameters cell accepts works here — env, tenant_id, and the
# control-table names if they differ from the olaf.* defaults.
#
# When:
#     once at the top of a session, only if your env / tenant / table names differ from the dev
#     defaults. Chainable; skip it entirely to run against the defaults.
# Expect:
#     returns a DataFrame of every parameter the next run will use, each row marked
#     set / default / per-call; every later OLAF.<action>() then uses these base params.
OLAF.configure(
    env="qa",
    tenant_id="00000000-0000-0000-0000-000000000000",  # "" = auto-resolve from the runtime context
    config_table="ml_gold.security_config",  # override any control-table name —
    log_table="ml_gold.security_log",  # they don't have to use the olaf.* defaults
)

# Just changing the env? One-liner: OLAF.configure(env="dev")

## Deployment

The `generate → plan → apply → rollback` chain. `generate` freezes the config into the mapping
lock-file; `plan` diffs it against the live DAR; `apply` pushes it; `rollback` restores a prior
config version. All return a compact DataFrame view of the outcome (`OLAF.last_result` = raw dict).

In [ ]:
# Preview the roles/scopes/predicates a config WOULD produce, BEFORE generate ever runs — a dry
# projection over generate's own resolution chain. No mapping/log write; it DOES read onelake_security_member —
# safe to call against a config you haven't committed to yet.
#
# When:
#     reviewing a config edit before generate/plan/apply — "what would this build?"
# Expect:
#     one row per role × scope grant (members = RESOLVED display names, ';'-joined — a glob: pattern shows its expansion); an
#     empty/all-inactive config → an empty but typed frame.
OLAF.explain()

#
# → Spark DataFrame (one row per role × scope grant; nothing written):
# +----------------+----------------------+------------+---------------+-----------------+---------------------+
# | role_name      | scope_path           | permission | rls_condition | visible_columns | members             |
# +----------------+----------------------+------------+---------------+-----------------+---------------------+
# | SalesReaders   | /Tables/sales/orders | Read       | null          | null            | sg-sales            |
# | FinanceReaders | /Tables/fin/ledger   | Read       | region='APAC' | amount;region   | priya@contoso.com |
# +----------------+----------------------+------------+---------------+-----------------+---------------------+

In [ ]:
# Build the mapping lock-file from the short config: resolve include/exclude, validate every
# rule, freeze the grants + write a versioned review CSV. rebuild=False is the normal path.
#
# When:
#     after every config edit — always before plan/apply. rebuild=False is idempotent, so
#     re-running an unchanged config is a no-op (status "skipped").
# Expect:
#     status "success" with the grants/roles/warnings counts (or "skipped" if unchanged),
#     plus a new versioned CSV directly under mapping_history_dir (default Files/security/mapping-history).
OLAF.generate(rebuild=False)

#
# → Spark DataFrame (OLAF._view of the generate envelope):
# +----------+---------+---------+--------------------------+--------+-------+----------+-------------------------------------------------------------------------------------------------+
# | mode     | status  | changed | message                  | grants | roles | warnings | csv                                                                                             |
# +----------+---------+---------+--------------------------+--------+-------+----------+-------------------------------------------------------------------------------------------------+
# | generate | success | true    | 12 grants across 3 roles | 12     | 3     | 1        | Files/security/mapping-history/onelake_security_mapping_20260715-080000_v7_a1b2c3d4e5f60718.csv |
# +----------+---------+---------+--------------------------+--------+-------+----------+-------------------------------------------------------------------------------------------------+

In [ ]:
# Diff the desired state (the mapping lock-file) against the live DAR — read-only, and it is
# what unlocks apply.
#
# When:
#     after generate, before apply — required (apply stays blocked until a matching plan exists
#     with no since-plan drift).
# Expect:
#     one row per change: create / update / omit / no_change. changed=false means live already
#     matches config, so apply is skipped.
OLAF.plan()

#
# → Spark DataFrame (one row per changed role):
# +------+---------+---------+-------------+----------------+--------+
# | mode | status  | changed | message     | role           | action |
# +------+---------+---------+-------------+----------------+--------+
# | plan | success | true    | 3 change(s) | FinanceReaders | update |
# | plan | success | true    | 3 change(s) | SalesReaders   | create |
# | plan | success | true    | 3 change(s) | TempAuditors   | omit   |
# +------+---------+---------+-------------+----------------+--------+

In [ ]:
# Push the planned roles to the live DAR. keep_unmanaged=True = INCREMENTAL upsert: managed roles
# are created/updated and NOTHING is dropped from the payload — hand-added Default* roles are
# carried through untouched.
#
# When:
#     the deliberate opt-in — you want live roles outside config left alone. NOT the default.
# Expect:
#     push_status (the bulk PUT's HTTP status, NOT a count) beside roles_written (the real count of
#     roles the PUT body carried); roles absent from config are listed under
#     drift_omission_candidates but are still carried in the submitted payload.
OLAF.apply(keep_unmanaged=True)

#
# → Spark DataFrame (OLAF._view of the apply envelope):
# +-------+---------+---------+---------------------+-------------+---------------+----------------+---------------------+------------------------------------------------------+-------------------------+---------------------------+----------------------------+
# | mode  | status  | changed | message             | push_status | roles_written | keep_unmanaged | request             | backup_path                                          | omitted_role_candidates | drift_omission_candidates | post_state_review_required |
# +-------+---------+---------+---------------------+-------------+---------------+----------------+---------------------+------------------------------------------------------+-------------------------+---------------------------+----------------------------+
# | apply | success | true    | apply (incremental) | 200         | 3             | true           | incremental_payload | Files/security/role-backups/onelake_..._replace.json | []                      | ["TempAuditors"]          | true                       |
# +-------+---------+---------+---------------------+-------------+---------------+----------------+---------------------+------------------------------------------------------+-------------------------+---------------------------+----------------------------+
#
# push_status is HTTP; roles_written is the count. Incremental PUTs live ∪ desired, so the 3 here is
# the 2 managed roles PLUS the kept TempAuditors — the whole body the PUT wrote.

In [ ]:
# The DEFAULT apply submits config as the whole truth: every live role NOT in the desired set is
# OMITTED from the payload, including hand-added Default* roles (the omitted list prints first).
# Omission is a REQUEST shape, not a confirmed outcome — the Preview bulk endpoint does not
# document deletion-by-omission, so review the post-state before concluding a role is gone.
#
# When:
#     routine deployment, right after you have reviewed the plan.
# Expect:
#     the omission-candidate list printed first (roles_written = the roles the PUT body carried,
#     omitted_role_candidates = the roles dropped by omission from it, and
#     post_state_review_required = true because the request alone settles nothing).
OLAF.apply()

#
# → Spark DataFrame:
# +-------+---------+---------+-----------------+-------------+---------------+----------------+----------------+------------------------------------------------------+-------------------------+---------------------------+----------------------------+
# | mode  | status  | changed | message         | push_status | roles_written | keep_unmanaged | request        | backup_path                                          | omitted_role_candidates | drift_omission_candidates | post_state_review_required |
# +-------+---------+---------+-----------------+-------------+---------------+----------------+----------------+------------------------------------------------------+-------------------------+---------------------------+----------------------------+
# | apply | success | true    | apply (replace) | 200         | 3             | false          | config_payload | Files/security/role-backups/onelake_..._replace.json | ["TempAuditors"]        | ["TempAuditors"]          | true                       |
# +-------+---------+---------+-----------------+-------------+---------------+----------------+----------------+------------------------------------------------------+-------------------------+---------------------------+----------------------------+
#
# If the PUT FAILS, OLAF.apply() still returns a frame — run_mode is the non-raising engine, so the
# view is the 1-row apply summary carrying status "error" (or "blocked" when a first-attempt 412
# refused the conditional PUT); the raw envelope stays at OLAF.last_result. Only the pipeline
# entrypoint run_and_exit raises on a failed outcome. The record is in
# onelake_security_log: one `failed` row per planned role (intended action vs PRESENT/ABSENT in a
# re-read of live) plus a `push` summary row carrying the intended payload and backup_path. Nothing
# is auto-restored: a PUT that timed out may have succeeded, so restoring is a deliberate act.

In [ ]:
# Restore the config table to a prior Delta version, then re-run generate → plan → apply. A
# blank version = the immediately previous one; a value pins an exact one. reason is required.
#
# When:
#     after a bad config edit you need to undo.
# Expect:
#     config restored to the target version, the full pipeline re-run, and two durable audit rows
#     (action "rollback", status "prepared" then "restored") whose message is a JSON record
#     carrying from_version, to_version, reason, source_config_hash and target_config_hash.
OLAF.rollback(rollback_to_version="41", rollback_reason="bad RLS predicate on Orders")

#
# → Spark DataFrame (1-row status summary):
# +----------+---------+---------+-----------------------------------------------------+
# | mode     | status  | changed | message                                             |
# +----------+---------+---------+-----------------------------------------------------+
# | rollback | success | true    | rollback to config v41: bad RLS predicate on Orders |
# +----------+---------+---------+-----------------------------------------------------+

In [ ]:
# Dry-run the IDENTICAL validation pipeline generate runs (every rule in the docs/architecture.md
# rule catalog, the No-Graph member gate, the lakehouse target guard) with ZERO writes -- no
# mapping, no CSV, no log row (not even a 'rejected' one on a blocked run, unlike generate). Safe
# to run against a live deployment: it never touches the control tables.
#
# When:
#     previewing a config edit's validation outcome before committing to generate (which writes).
# Expect:
#     status "success" with the grant/role/warning summary on a clean config; on an invalid config
#     a blocked view carrying the SAME collect-all error set generate would reject with (never
#     raises — the raw envelope, with counts + every warning, stays at OLAF.last_result).
OLAF.validate()

#
# → Spark DataFrame (OLAF._view of the validate envelope — the 1-row status summary; the raw
#    envelope with the grant/role counts + every warning stays at OLAF.last_result):
# +----------+---------+---------+---------------------------------------------------------------------------+
# | mode     | status  | changed | message                                                                   |
# +----------+---------+---------+---------------------------------------------------------------------------+
# | validate | success | false   | validate: 12 grant(s) across 3 role(s), 1 warning(s) — dry-run, no writes |
# +----------+---------+---------+---------------------------------------------------------------------------+

## Audit

Read-only queries over the live DAR + the `onelake_security_log` trail — no SQL. `show`/`trace`
route through `run_mode` (they need the live target); `grants` / `provenance` / `timeline` /
`report` read the log directly. All return a Spark DataFrame.

In [ ]:
# Pivot the LIVE DAR by TABLE: who can reach sales.orders, each grant enriched from the log
# (first_applied/first_granted_by · last_applied/last_granted_by · config_version) and flagged
# framework vs out-of-band (a Fabric-UI edit).
#
# When:
#     answering "who can access this table?" — audit / compliance / incident review, any time (read-only).
# Expect:
#     one row per (role, member) reaching the table, with the enrichment + an out_of_band flag.
OLAF.show(by="table", subject="sales.orders")

#
# → Spark DataFrame (one row per role × scope × member; member = objectId, abbreviated here).
# by=table LEADS with scope_path — every axis returns the same eleven columns, led by its own key:
# +----------------------+--------------+-----------+-------------------+------------+---------------+------------------+--------------+-----------------+----------------+------------+
# | scope_path           | role_name    | member    | member_name       | permission | first_applied | first_granted_by | last_applied | last_granted_by | config_version | provenance |
# +----------------------+--------------+-----------+-------------------+------------+---------------+------------------+--------------+-----------------+----------------+------------+
# | /Tables/sales/orders | SalesReaders | 8f4e2a…c1 | priya@contoso.com | Read       | 2026-07-15    | ci@contoso       | 2026-08-24   | keng@contoso    | 42             | framework  |
# +----------------------+--------------+-----------+-------------------+------------+---------------+------------------+--------------+-----------------+----------------+------------+

In [ ]:
# Pivot the LIVE DAR by ROLE: every table/folder scope + member that SalesReaders grants.
#
# When:
#     reviewing one role — "what does SalesReaders actually grant?"
# Expect:
#     one row per (scope, member) the role covers, with the same log enrichment as by=table.
OLAF.show(by="role", subject="SalesReaders")

#
# → Spark DataFrame (same eleven columns, led by role_name; one row per scope × member of the role):
# +--------------+-----------------------+-----------+-------------------+------------+---------------+------------------+--------------+-----------------+----------------+------------+
# | role_name    | scope_path            | member    | member_name       | permission | first_applied | first_granted_by | last_applied | last_granted_by | config_version | provenance |
# +--------------+-----------------------+-----------+-------------------+------------+---------------+------------------+--------------+-----------------+----------------+------------+
# | SalesReaders | /Tables/sales/orders  | 8f4e2a…c1 | priya@contoso.com | Read       | 2026-07-15    | ci@contoso       | 2026-08-24   | keng@contoso    | 42             | framework  |
# | SalesReaders | /Tables/sales/returns | 8f4e2a…c1 | priya@contoso.com | Read       | 2026-07-15    | ci@contoso       | 2026-08-24   | keng@contoso    | 42             | framework  |
# +--------------+-----------------------+-----------+-------------------+------------+---------------+------------------+--------------+-----------------+----------------+------------+

In [ ]:
# Pivot the LIVE DAR by MEMBER: every role + scope priya@contoso.com reaches
# (accepts an objectId or a name / glob).
#
# When:
#     an access review for one person — "what can they see?"
# Expect:
#     one row per (role, scope) the member reaches, enriched + out_of_band-flagged.
OLAF.show(by="member", subject="priya@contoso.com")

#
# → Spark DataFrame (same eleven columns; by=member LEADS with member + member_name):
# +-----------+-------------------+----------------+----------------------+------------+---------------+------------------+--------------+-----------------+----------------+------------+
# | member    | member_name       | role_name      | scope_path           | permission | first_applied | first_granted_by | last_applied | last_granted_by | config_version | provenance |
# +-----------+-------------------+----------------+----------------------+------------+---------------+------------------+--------------+-----------------+----------------+------------+
# | 8f4e2a…c1 | priya@contoso.com | SalesReaders   | /Tables/sales/orders | Read       | 2026-07-15    | ci@contoso       | 2026-08-24   | keng@contoso    | 42             | framework  |
# | 8f4e2a…c1 | priya@contoso.com | FinanceReaders | /Tables/fin/ledger   | Read       | 2026-07-16    | ci@contoso       | 2026-08-24   | keng@contoso    | 42             | framework  |
# +-----------+-------------------+----------------+----------------------+------------+---------------+------------------+--------------+-----------------+----------------+------------+

In [ ]:
# Operational snapshot (the trace mode / Audit.report): the deployed generation, LIVE role +
# grant counts, staleness, and — when the live client resolves — the out-of-band grant count.
# Every count describes the CURRENT state except established_ever, which is cumulative across
# every env and config version and is named so it cannot be mistaken for today.
#
# When:
#     a quick "is the deployment healthy / in sync?" check, any time (read-only).
# Expect:
#     a 1-row snapshot: generation, live role/grant counts, is_stale, out_of_band count.
OLAF.trace()

#
# → Spark DataFrame (1-row snapshot):
# +-------+---------+-----------------+------------------+------------------+----------+-------------+
# | mode  | status  | live_role_count | live_grant_count | established_ever | is_stale | out_of_band |
# +-------+---------+-----------------+------------------+------------------+----------+-------------+
# | trace | success | 3               | 7                | 19               | false    | 1           |
# +-------+---------+-----------------+------------------+------------------+----------+-------------+

In [ ]:
# Established grants read from the LOG (no live DAR call) — one row per (role, scope, member)
# with provenance. Optional role / scope / member narrow the listing.
#
# When:
#     "what has the framework granted, and when?" — from the audit trail, offline.
# Expect:
#     one row per established (role, scope, member), each with BOTH ends of its deploy range:
#     first_applied/first_granted_by, last_applied/last_granted_by, and config_version.
OLAF.grants(role="SalesReaders")

#
# → Spark DataFrame:
# +--------------+----------------------+-----------+-------------------+---------------+------------------+--------------+-----------------+----------------+
# | role_name    | scope_path           | member_id | member_name       | first_applied | first_granted_by | last_applied | last_granted_by | config_version |
# +--------------+----------------------+-----------+-------------------+---------------+------------------+--------------+-----------------+----------------+
# | SalesReaders | /Tables/sales/orders | 8f4e2a…c1 | priya@contoso.com | 2026-07-15    | ci@contoso       | 2026-08-24   | keng@contoso    | 42             |
# +--------------+----------------------+-----------+-------------------+---------------+------------------+--------------+-----------------+----------------+

In [ ]:
# One established grant's provenance for a (role, scope[, member]) — the first row of grants(),
# as a 1-row DataFrame (or an empty frame when no such grant exists).
#
# Reports BOTH ends, and deliberately no single `since`. The log records what OLAF did; it
# cannot see a role deleted straight from the Fabric UI, so an apply, an out-of-band deletion
# and a re-apply are indistinguishable here from one unbroken grant. Read first_applied as
# "first deployed on" and last_applied as "last re-asserted on" — a wide gap between them is a
# cue to check the live DAR, not evidence either way.
#
# When:
#     an access review — "when did this access start, and has anyone touched it since?"
# Expect:
#     a 1-row frame with both ends and the principal who pushed each (or an empty frame if none).
OLAF.provenance(role="SalesReaders", scope="/Tables/sales/orders")

#
# → Spark DataFrame (1 row):
# +--------------+----------------------+-----------+-------------------+---------------+------------------+--------------+-----------------+----------------+
# | role_name    | scope_path           | member_id | member_name       | first_applied | first_granted_by | last_applied | last_granted_by | config_version |
# +--------------+----------------------+-----------+-------------------+---------------+------------------+--------------+-----------------+----------------+
# | SalesReaders | /Tables/sales/orders | 8f4e2a…c1 | priya@contoso.com | 2026-07-15    | ci@contoso       | 2026-08-24   | keng@contoso    | 42             |
# +--------------+----------------------+-----------+-------------------+---------------+------------------+--------------+-----------------+----------------+

In [ ]:
# Every logged config generation as one row: first_seen / last_seen / run count per
# (config_version, config_hash), ordered by version — the lifetime of each generation.
#
# When:
#     reviewing how the config has evolved over time — its change history.
# Expect:
#     one row per generation: config_version, config_hash, first_seen, last_seen, runs.
OLAF.timeline()

#
# → Spark DataFrame:
# +----------------+-------------+---------------------+---------------------+------+
# | config_version | config_hash | first_seen          | last_seen           | runs |
# +----------------+-------------+---------------------+---------------------+------+
# | 41             | 3b9c…       | 2026-07-10T09:12:03 | 2026-07-14T18:40:55 | 6    |
# | 42             | a17f…       | 2026-07-15T08:01:22 | 2026-07-16T11:29:07 | 4    |
# +----------------+-------------+---------------------+---------------------+------+

In [ ]:
# The one-call operational snapshot behind trace, as a 1-row DataFrame. Nested values (the
# generation / last-run dicts) are stringified; the counts and is_stale are scalar.
#
# When:
#     same as trace, when you want the full dict fields in one row.
# Expect:
#     a 1-row wide frame: current_generation, last_generate, last_apply, live counts,
#     established_ever, is_stale, out_of_band.
OLAF.report()

#
# → Spark DataFrame (1 row; wide — nested dicts shown abbreviated):
# +---------------------------+------------------+----------------+-----------------+------------------+------------------+----------+-------------+
# | current_generation        | last_generate    | last_apply     | live_role_count | live_grant_count | established_ever | is_stale | out_of_band |
# +---------------------------+------------------+----------------+-----------------+------------------+------------------+----------+-------------+
# | {'config_version': 42, …} | {'mode': 'gene…} | {'mode': 'ap…} | 3               | 7                | 19               | false    | 1           |
# +---------------------------+------------------+----------------+-----------------+------------------+------------------+----------+-------------+

### Compliance & live-DAR utilities

Read-only queries that pivot the LIVE DAR directly (all but `coverage` need a live client) —
finding gaps, computing net access, and comparing desired vs. live without touching `plan`/`apply`.

In [ ]:
# Protected vs unprotected table surface — the compliance gap finder. The table universe is every
# REAL table in the lakehouse catalog, not just the ones named in the mapping, so a table nobody
# configured at all still gets a row (protected=false) instead of being silently skipped.
#
# When:
#     "what tables have NO security applied at all?" — a compliance sweep, any time (no live
#     client needed).
# Expect:
#     one row per catalog table; protected=false + roles_count=0 for anything unconfigured.
OLAF.coverage()

#
# → Spark DataFrame (one row per table in the catalog):
# +-----------------------+-----------+-------------+---------+---------+
# | table                 | protected | roles_count | has_rls | has_cls |
# +-----------------------+-----------+-------------+---------+---------+
# | sales.orders          | true      | 1           | true    | false   |
# | fin.ledger            | true      | 1           | true    | true    |
# | sales.staging_scratch | false     | 0           | false   | false   |
# +-----------------------+-----------+-------------+---------+---------+

In [ ]:
# Net EFFECTIVE access to one table for one member — most-permissive-wins across every reaching
# role, plus a synthesized 'effective' union row. engine= is REQUIRED (spark | direct_lake |
# sql_endpoint) and is echoed as a column: on spark/direct_lake a reaching role with no RLS/CLS on
# the table nullifies every other role's restriction, mirroring rule C8; on sql_endpoint CLS is the
# INTERSECTION of the explicit allow-lists instead, so an unrestricted role does NOT erase another
# role's column restriction (RLS is still nullified on every engine). This models the DAR rules —
# it is not proof of endpoint identity mode or of enforcement for a given request.
# Needs a live client (Fabric only).
# member accepts a name/UPN (resolved case-insensitively against onelake_security_member, the
# same No-Graph table generate resolves from) or an objectId (GUID) passed straight through; a
# name absent from the member table raises UsageError naming the No-Graph limitation. The optional
# member_type ("Group"/"User"/"ServicePrincipal"/"ManagedIdentity") scopes resolution to the member
# table's logical PK (member_type, lower(member_name)) — needed only when a Group and a User share a
# display name, which is otherwise a hard UsageError naming the ambiguity instead of a silent pick.
#
# When:
#     "what can this person actually see on this table, once every role they're in is combined?"
# Expect:
#     one detail row per reaching role, plus one effective=true union row; no reaching role → an
#     empty but typed frame.
OLAF.effective_access(member="priya@contoso.com", table="sales.orders", engine="spark")
# member is a name/UPN preloaded in onelake_security_member — a GUID objectId also works unchanged

#
# → Spark DataFrame (one row per reaching role, plus the union):
# +----------------+---------------+-----------------+-----------------------------+-----------+--------+
# | role_name      | rls_condition | visible_columns | granting_role               | effective | engine |
# +----------------+---------------+-----------------+-----------------------------+-----------+--------+
# | SalesReaders   | region='APAC' | null            | null                        | false     | spark  |
# | FinanceReaders | null          | null            | null                        | false     | spark  |
# | null           | null          | null            | FinanceReaders;SalesReaders | true      | spark  |
# +----------------+---------------+-----------------+-----------------------------+-----------+--------+
# (FinanceReaders carries no RLS on this table, so the union's rls_condition is null too —
#  "unrestricted nullifies filter".)

# Same name, two principal types (a Group AND a User called "finance-team") — say which one:
OLAF.effective_access(
    member="finance-team", table="sales.orders", member_type="Group", engine="spark"
)

In [ ]:
# The REVERSE of effective_access — every member who can reach this table, one row per
# (member, role) pair (a member reachable via two roles gets two rows). Needs a live client.
#
# When:
#     "who can see this table, and through which role?" — an access review by table.
# Expect:
#     one row per (member, role); rls_cls_summary is a one-cell digest ('rows: ...' / 'cols: ...' /
#     'unrestricted').
OLAF.who_can_access(table="sales.orders")

#
# → Spark DataFrame:
# +---------------------+-----------+----------------+------------+---------------------+
# | member_name         | member_id | via_role       | permission | rls_cls_summary     |
# +---------------------+-----------+----------------+------------+---------------------+
# | priya@contoso.com | 8f4e2a…c1 | SalesReaders   | Read       | rows: region='APAC' |
# | sg-finance-leads    | c9d1e0…7f | FinanceReaders | Read       | unrestricted        |
# +---------------------+-----------+----------------+------------+---------------------+

In [ ]:
# Full desired-vs-live DAR comparison, CATEGORIZED and READ-ONLY: framework (matches provenance) /
# out_of_band (no provenance) / policy (provenanced, but the deployed permission / RLS predicate /
# CLS allow-list differs from what the mapping declares — out_of_band WINS when a grant qualifies
# for both) / missing (desired but absent live). member_id + member_name are
# APPENDED at the end of the frame (out_of_band()'s convention); member_name is resolved id->name
# from the member cache table (same lookup out_of_band/who_can_access use) and falls back to the id
# itself when the cache has no row — which is exactly what the id/name PAIR lets you detect. A third
# rendering exists: an id the cache gives MORE THAN ONE distinct name surfaces as
# "<ambiguous: N names in member table>", distinct from a real name AND from the bare-id fallback.
# A pure
# comparison view — it never records a plan and never gates apply (plan() stays the one
# apply-gating recorder). Needs a live client.
#
# When:
#     a periodic "is live DAR exactly what config says?" check, independent of plan/apply.
# Expect:
#     one row per grant (or per missing desired grant), tagged with its category + a short detail.
OLAF.drift()

#
# → Spark DataFrame:
# +----------------+----------------------+-------------+-----------------------------------------------+-----------+---------------------+
# | role_name      | scope_path           | category    | detail                                        | member_id | member_name         |
# +----------------+----------------------+-------------+-----------------------------------------------+-----------+---------------------+
# | SalesReaders   | /Tables/sales/orders | framework   | live grant matches framework provenance (...) | 8f4e2a…c1 | priya@contoso.com |
# | FinanceReaders | /Tables/fin/ledger   | out_of_band | live grant has no framework provenance (...)  | c9d1e0…7f | sg-finance-leads    |
# | TempAuditors   | /Tables/sales/orders | missing     | desired grant absent from live DAR (...)      | a2b3c4…5d | sg-temp-auditors    |
# +----------------+----------------------+-------------+-----------------------------------------------+-----------+---------------------+

### Config time-travel & lineage

Delta version-history utilities over the control tables — `table_history`/`at` work on any of
config/mapping/log; `config_diff`/`value_history` are config-specific, diffing the authored rows
across versions.

In [ ]:
# Delta DESCRIBE HISTORY of a control table, made readable. `table` is one of config/mapping/log
# (mapped to the actual configured table name).
#
# When:
#     "when did the config table last change, and who/what wrote it?" — a Delta-native history,
#     independent of onelake_security_log.
# Expect:
#     one row per Delta commit, newest first (DESCRIBE HISTORY's own order).
OLAF.table_history("config")

#
# → Spark DataFrame:
# +---------+---------------------------+-------------------+-----------+------+
# | version | timestamp                 | user              | operation | rows |
# +---------+---------------------------+-------------------+-----------+------+
# | 42      | 2026-07-15T08:00:00+00:00 | alice@contoso.com | UPDATE    | 14   |
# | 41      | 2026-07-10T09:00:00+00:00 | alice@contoso.com | UPDATE    | 13   |
# +---------+---------------------------+-------------------+-----------+------+

In [ ]:
# Snapshot of ANY control table (config/mapping/log) at a Delta version or date — config_at
# generalized to every control table. Give exactly one of version or date.
#
# When:
#     "what did the mapping lock-file look like right after the v41 rollback?" — any control
#     table, any point in time.
# Expect:
#     the resolved table's own schema, as of that version/date.
OLAF.at("mapping", version=7)

#
# → Spark DataFrame (onelake_security_mapping's own schema, as of version 7 — abbreviated here):
# +--------------+----------------------+------------+-----+
# | role_name    | scope_path           | permission | ... |
# +--------------+----------------------+------------+-----+
# | SalesReaders | /Tables/sales/orders | Read       | ... |
# +--------------+----------------------+------------+-----+

In [ ]:
# Role/scope/member changes between two config Delta versions — added / removed / changed rows,
# diffed in Python after two time-travel reads (config_at(v1), config_at(v2)).
#
# When:
#     "what actually changed between config v41 and v42?" — reviewing an edit after the fact.
# Expect:
#     one row per added/removed role×scope key, or one row per changed field (a role with 3
#     changed fields → 3 'changed' rows).
OLAF.config_diff(41, 42)

#
# → Spark DataFrame (scope_key is the config's own include/exclude-column key, '|'-joined):
# +-------------+--------------+-------------------+---------------+-------+---------------+
# | change_type | role_name    | scope_key         | field         | old   | new           |
# +-------------+--------------+-------------------+---------------+-------+---------------+
# | changed     | SalesReaders | sales.orders|||   | rls_condition | null  | region='APAC' |
# | added       | TempAuditors | sales.orders|||   | null          | null  | null          |
# +-------------+--------------+-------------------+---------------+-------+---------------+

In [ ]:
# How ONE role/scope's config VALUE evolved across every config Delta version — walks every
# version, keeping only the rows where the subject is present, flagging `changed` on its first
# appearance and on any later version where a tracked field differs from the last one it appeared in.
#
# When:
#     "when did SalesReaders' RLS predicate actually change, across every version?" — a full
#     lineage for one role/scope, not just a two-version diff (that's config_diff).
# Expect:
#     one row per version where the subject is present; changed=true on first appearance.
OLAF.value_history(subject="SalesReaders")

#
# → Spark DataFrame (20 columns total — every config_diff field column, abbreviated here):
# +----------------+--------------+-----------------+------------+---------------+-----+---------+------------------+
# | config_version | role_name    | scope_key       | permission | rls_condition | ... | changed | window_truncated |
# +----------------+--------------+-----------------+------------+---------------+-----+---------+------------------+
# | 41             | SalesReaders | sales.orders||| | Read       | null          | ... | true    | false            |
# | 42             | SalesReaders | sales.orders||| | Read       | region='APAC' | ... | true    | false            |
# +----------------+--------------+-----------------+------------+---------------+-----+---------+------------------+

## Maintenance

Control-table lifecycle. `setup` is the only maintenance action on `OLAF` today.

In [ ]:
# Create (or additively migrate) the four control tables — config / mapping / log / member.
# Idempotent: safe to re-run; it reconciles schema drift without data loss.
#
# setup REQUIRES lakehouse_name, and it is an assertion, not a target: setup always writes to the
# ATTACHED lakehouse (its DDL uses two-part `olaf.…` names), so naming it here is how you get told
# you are attached to the wrong one — instead of finding the control tables in another workspace
# later. Case-insensitive; an attachment from another workspace is refused on the ids too.
#
# When:
#     the first thing you run in a new lakehouse, and again after a framework upgrade.
# Expect:
#     the four tables created/migrated + a 1-row status summary; a `blocked` row naming BOTH the
#     declared and the attached lakehouse if they disagree.
OLAF.configure(lakehouse_name="LH_Gold")  # the lakehouse this notebook is attached to
OLAF.setup()

#
# → Spark DataFrame (1-row status summary):
# +-------+---------+---------+-----------------------------------+
# | mode  | status  | changed | message                           |
# +-------+---------+---------+-----------------------------------+
# | setup | success | true    | created/migrated 4 control tables |
# +-------+---------+---------+-----------------------------------+

In [ ]:
# One-call doctor: 9 independent checks (control tables present, table location, mapping
# staleness, DAR reachability, control-data exposure, identity preflight, runtime prerequisites,
# last-apply age, out-of-band grants). One failing check never aborts the rest — health() always
# returns all 9 rows.
#
# When:
#     "is everything healthy?" — a quick doctor pass, any time (never raises).
# Expect:
#     always 9 rows, each pass/warn/fail with a human-readable detail.
OLAF.health()

#
# → Spark DataFrame (always 9 rows):
# +-----------------------+--------+----------------------------------------------------------------------+
# | check                 | status | detail                                                               |
# +-----------------------+--------+----------------------------------------------------------------------+
# | control_tables        | pass   | all 4 control tables present with the expected schema                |
# | table_location        | pass   | control tables are in the attached lakehouse (LH_Gold)               |
# | mapping_staleness     | pass   | mapping matches the active config (version 42)                       |
# | dar_reachable         | pass   | bounded DAR read succeeded with a collection ETag                    |
# | control_data_exposure | pass   | JSON facts show a safe DAR snapshot and workspace_isolation=attested |
# | identity_preflight    | pass   | Fabric token acquired for the ambient identity (alice@...)           |
# | runtime_prerequisites | pass   | observed Spark meets the baseline; verify the Fabric Runtime label   |
# | last_apply_age        | pass   | last apply was 2 day(s) ago                                          |
# | out_of_band           | warn   | 1 out-of-band grant(s) with no framework provenance                  |
# +-----------------------+--------+----------------------------------------------------------------------+

In [ ]:
# One-call at-a-glance deployment snapshot, built purely from the log + mapping (no live client
# needed, unlike health()'s DAR-dependent checks).
#
# When:
#     "what's deployed right now, in one row?" — a lightweight dashboard check.
# Expect:
#     a 1-row frame: role/member counts, last generate/apply timestamps, the newest successful
#     deployment and its mode, the live config version, and whether an unapplied change is pending.
OLAF.status()

#
# → Spark DataFrame (1 row):
# +---------+-----------+---------------------------+---------------------------+---------------------------+----------------------+---------------------+----------------+
# | n_roles | n_members | last_generate             | last_apply                | last_deployment           | last_deployment_mode | live_config_version | pending_change |
# +---------+-----------+---------------------------+---------------------------+---------------------------+----------------------+---------------------+----------------+
# | 3       | 5         | 2026-07-15T08:00:00+00:00 | 2026-07-15T08:05:00+00:00 | 2026-07-15T08:05:00+00:00 | apply                | 42                  | false          |
# +---------+-----------+---------------------------+---------------------------+---------------------------+----------------------+---------------------+----------------+

In [ ]:
# Why can't `member` see data? Walks the same chain a human troubleshooter would, IN ORDER. Once a
# prerequisite step (member_in_table / id_resolved / in_mapping) fails, every later step is
# short-circuited with "skipped — prerequisite failed" instead of reporting a misleading guess.
#
# When:
#     "why doesn't priya see the data they should?" — a single, ordered root-cause walk.
# Expect:
#     always 5 rows, in order: member_in_table → id_resolved → in_mapping → live_in_dar →
#     apply_in_sync.
OLAF.diagnose_member("priya@contoso.com")
# An objectId works too — GUID pass-through, the same one effective_access gives (step 1 reports
# the pass-through instead of a name lookup), so an id copied out of who_can_access() is accepted:
OLAF.diagnose_member("8f4e2a00-0000-0000-0000-0000000000c1")

#
# → Spark DataFrame (always 5 rows, in order):
# +-----------------+------+--------------------------------------------------------------+
# | step            | ok   | detail                                                       |
# +-----------------+------+--------------------------------------------------------------+
# | member_in_table | true | found in olaf.onelake_security_member (type User)            |
# | id_resolved     | true | resolved to objectId 8f4e2a…c1                               |
# | in_mapping      | true | in role(s): SalesReaders                                     |
# | live_in_dar     | true | live in role(s): SalesReaders                                |
# | apply_in_sync   | true | mapping is in sync with the last apply (2026-07-15T08:05:00) |
# +-----------------+------+--------------------------------------------------------------+

## 🔥 Destructive — read before running either cell below

> ### These two are destructive. There is no dry run, no review gate, and no undo prompt.
> The moment the cell finishes, it is done.

| | `OLAF.reset()` | `OLAF.cleanup()` |
|---|---|---|
| **does** | submits an **empty DAR payload**, omitting **every** live role — OLAF's, other people's, and `Default*` | deletes all four control tables (**including the whole audit log**) + **every** mapping-history and role-backup file |
| **leaves** | the control tables — config, mapping, member, log | the live roles, now with no trail explaining them |
| **way back** | the pre-request backup it writes first — a recovery *input*, not a guaranteed exact restore, and there is no public replay method: recovery is a break-glass incident (RUNBOOK §3c) | **none. It deletes the backups too.** |

**`reset()` submits a request; it does not report an outcome.** The roles it returns are
`prior_live_role_candidate`s — observed before submission and omitted from the request. The Preview
bulk endpoint does not document deletion-by-omission, so OLAF will not tell you the roles are gone,
that OneLake security is now deny-by-default, or that nobody can read the data. `post_state_review_required`
comes back **true**: re-read the post-state in the target engine and access mode before drawing any
access conclusion. Note the platform's `DefaultReader` is among the omitted roles and OLAF does
**not** recreate platform-managed roles.

⚠️ **Do not assume a uniform outcome across principals** — workspace roles, default roles, engine/access
mode and shortcut behavior all remain relevant, and OLAF claims neither universal enforcement nor a
universal privileged-role bypass. Verify the post-state in each engine/access mode you actually care about.

⚠️ **`cleanup()` cannot log what it did** — it drops the log table. The returned frame and the
printed lines are the only record that run will ever produce.

Neither is a mode: a pipeline passing `mode="reset"` is refused by name. They exist only here.

In [ ]:
# 🔥 DESTRUCTIVE containment request. SUBMITS AN EMPTY DAR PAYLOAD, omitting every data access
# role on the attached lakehouse — OLAF's own, anyone else's, and Default*. OLAF does not
# recreate platform-managed/default roles.
#
# What comes back are CANDIDATES, not confirmed deletions: roles observed before submission and
# omitted from the request. The Preview contract does not establish deletion-by-omission,
# no-OneLake-security, or universal-reader outcomes, so post_state_review_required is true —
# review the post-state in the target engine/access mode before concluding anything about access.
#
# Writes a role backup FIRST; if that capture fails the reset aborts before submitting anything.
# The backup is a recovery INPUT, not a guarantee of exact platform-state restoration (RUNBOOK 3c).
# Control tables are untouched, so generate -> plan -> apply can submit the config's payload again.
#
# When:
#     handing a lakehouse back, or starting its security over from nothing.
# Review the post-state afterwards in the engine/access mode you care about — OLAF claims neither
# universal enforcement nor a universal privileged-role bypass.
OLAF.reset()

#
# → Spark DataFrame — one row per prior-live role candidate, with the request label, the backup
#   artifact, and the review flag:
# +---------------------------+---------------+----------------------------------------------------+----------------------------+
# | prior_live_role_candidate | request       | backup_path                                        | post_state_review_required |
# +---------------------------+---------------+----------------------------------------------------+----------------------------+
# | DefaultReader             | empty_payload | Files/security/role-backups/onelake_..._reset.json | true                       |
# | SalesReaders              | empty_payload | Files/security/role-backups/onelake_..._reset.json | true                       |
# +---------------------------+---------------+----------------------------------------------------+----------------------------+

In [ ]:
# 🔥 THE ONE WITH NO WAY BACK. Drops all four control tables — the authored config, the mapping,
# the member table and the ENTIRE AUDIT HISTORY — and deletes every mapping-history CSV and every
# pre-apply role backup. Those backups are the recovery for a bad apply and for reset(); after
# this there is nothing left to restore from.
#
# It does NOT touch the live roles. Any still deployed are listed in the result, now with no audit
# trail behind them — run reset() FIRST if you want them out of the DAR payload, because
# afterwards its backup is gone too.
#
# The frame also carries what cleanup could NOT do: items it failed to remove, the incident
# sentinel it deliberately preserves, and an explicit exposure-not-remediated row. Containment
# is not proof of erasure, and the manifest says so rather than reading as an all-clear.
#
# It cannot log what it did (it drops the log). The frame below and the printed lines are the ONLY
# record. Copy them somewhere before closing the notebook.
#
# When:
#     first-time deploy into an environment a trial run left dirty, so setup() starts from nothing.
OLAF.cleanup()

#
# → Spark DataFrame — the manifest, and the only record this run produces:
# +-----------------------------+------------------------------------------------------------------+
# | kind                        | name                                                             |
# +-----------------------------+------------------------------------------------------------------+
# | dropped table               | olaf.onelake_security_config                                     |
# | dropped table               | olaf.onelake_security_log                                        |
# | deleted file                | Files/security/role-backups/onelake_..._replace.json             |
# | LIVE ROLE LEFT BEHIND       | SalesReaders                                                     |
# | INCIDENT SENTINEL PRESERVED | Files/security/.olaf-sensitive-write.sentinel                    |
# | EXPOSURE NOT REMEDIATED     | exposure_remediated=false; cleanup cannot retract prior reads, … |
# +-----------------------------+------------------------------------------------------------------+